<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این مثال از یک Vector Database واقعی به نام Qdrant استفاده می‌کنیم.

برای اینکه نیازی به نصب یا راه‌اندازی سرور نداشته باشی، از حالت In-Memory آن استفاده می‌کنیم.

سناریو: فقط اسناد مربوط به واحد HR، دسته‌بندی Leave و سال ۲۰۲۶ را جستجو می‌کنیم.

In [2]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

# 1. Initialize Embedding Model
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vector_size = embedding_model.get_embedding_dimension()

# 2. Initialize Qdrant In-Memory
client = QdrantClient(":memory:")

collection_name = "hr_documents"

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE
    )
)

# 3. HR Documents + Metadata
documents = [
    {
        "id": 1,
        "text": "Employees receive 26 annual leave days.",
        "metadata": {
            "department": "HR",
            "category": "Leave",
            "year": 2026
        }
    },
    {
        "id": 2,
        "text": "Leave requests must be approved by the manager.",
        "metadata": {
            "department": "HR",
            "category": "Leave",
            "year": 2026
        }
    },
    {
        "id": 3,
        "text": "Employees have supplementary health insurance.",
        "metadata": {
            "department": "HR",
            "category": "Insurance",
            "year": 2026
        }
    },
    {
        "id": 4,
        "text": "Monthly salaries are paid at the end of each month.",
        "metadata": {
            "department": "Finance",
            "category": "Salary",
            "year": 2026
        }
    },
    {
        "id": 5,
        "text": "Employees receive 24 annual leave days.",
        "metadata": {
            "department": "HR",
            "category": "Leave",
            "year": 2024
        }
    }
]

# 4. Create Payload Indexes
for field, field_type in [
    ("department", models.PayloadSchemaType.KEYWORD),
    ("category", models.PayloadSchemaType.KEYWORD),
    ("year", models.PayloadSchemaType.INTEGER)
]:
    client.create_payload_index(
        collection_name=collection_name,
        field_name=field,
        field_schema=field_type
    )

# 5. Generate Embeddings and Store Documents
points = []

for document in documents:
    vector = embedding_model.encode(
        document["text"],
        normalize_embeddings=True
    ).tolist()

    points.append(
        models.PointStruct(
            id=document["id"],
            vector=vector,
            payload={
                "text": document["text"],
                **document["metadata"]
            }
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points
)

# 6. Metadata-Based Semantic Search
def search_documents(query, top_k=5):

    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    search_results = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="department",
                    match=models.MatchValue(value="HR")
                ),
                models.FieldCondition(
                    key="category",
                    match=models.MatchValue(value="Leave")
                ),
                models.FieldCondition(
                    key="year",
                    range=models.Range(
                        gte=2026,
                        lte=2026
                    )
                )
            ]
        ),
        limit=top_k,
        with_payload=True
    ).points

    return search_results


# 7. Execute Search
query = "How many annual leave days do employees receive?"

results = search_documents(query)

print("Query:", query)
print("\nRetrieved Documents:")

for rank, result in enumerate(results, start=1):
    print(f"\nRank: {rank}")
    print("ID:", result.id)
    print("Text:", result.payload["text"])
    print("Department:", result.payload["department"])
    print("Category:", result.payload["category"])
    print("Year:", result.payload["year"])
    print("Score:", round(result.score, 4))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1017.47it/s]
C:\Users\am.mousavi\AppData\Local\Temp\ipykernel_1252\241320294.py:79: UserWarning: Payload indexes have no effect in the local Qdrant, except text indexes with `scoring`, which enable text queries. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


Query: How many annual leave days do employees receive?

Retrieved Documents:

Rank: 1
ID: 1
Text: Employees receive 26 annual leave days.
Department: HR
Category: Leave
Year: 2026
Score: 0.7969

Rank: 2
ID: 2
Text: Leave requests must be approved by the manager.
Department: HR
Category: Leave
Year: 2026
Score: 0.4278


<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این مثال چه اتفاقی افتاد؟
- ابتدا Embedding اسناد تولید شد.
- متن و Metadata هر سند در Qdrant ذخیره شدند.
- برای فیلدهای مورد استفاده در فیلتر، Payload Index ساختیم.
- هنگام جستجو، Qdrant فقط اسنادی را بررسی کرد که هر سه شرط Metadata را داشته باشند.
- از بین اسناد باقی‌مانده، نتایج بر اساس شباهت معنایی رتبه‌بندی شدند.
در Qdrant، شرط‌های داخل must به‌صورت AND ترکیب می‌شوند. همچنین برای جستجوی سریع‌تر در دیتاست‌های بزرگ، مستندات رسمی ایجاد Payload Index برای فیلدهای پرتکرار در فیلتر را توصیه می‌کنند.